# Explore here

In [ ]:
#dividir la carpeta de train en nuna carpeta de dogs y otra carpeta de cats
import os
import shutil

def split_train_folder(train_folder):
    dogs_folder = os.path.join(train_folder, 'dogs')
    cats_folder = os.path.join(train_folder, 'cats')

    # Crear las carpetas de dogs y cats si no existen
    os.makedirs(dogs_folder, exist_ok=True)
    os.makedirs(cats_folder, exist_ok=True)

    # Iterar sobre los archivos en la carpeta de train
    for filename in os.listdir(train_folder):
        #evita la carpeta dogs y cats
        if filename in ['dogs', 'cats']:
            continue
        if filename.startswith('dog'):
            shutil.move(os.path.join(train_folder, filename), os.path.join(dogs_folder, filename))
        elif filename.startswith('cat'):
            shutil.move(os.path.join(train_folder, filename), os.path.join(cats_folder, filename))

#ejemplo de uso
train_folder = r"C:\Users\anton\nn\nncatsdogs\data\raw\dogs-vs-cats\train"
split_train_folder(train_folder)


In [ ]:
#genera una carpeta de test seleccionando 1250 imagenes de cada clase de la carpeta de train y moviendolas a la carpeta de test
import random

def create_test_folder(train_folder, test_folder, num_samples=1250):
    dogs_folder = os.path.join(train_folder, 'dogs')
    cats_folder = os.path.join(train_folder, 'cats')

    test_dogs_folder = os.path.join(test_folder, 'dogs')
    test_cats_folder = os.path.join(test_folder, 'cats')
    os.makedirs(test_dogs_folder, exist_ok=True)
    os.makedirs(test_cats_folder, exist_ok=True)
    
    dog_files = [f for f in os.listdir(dogs_folder) if f.endswith('.jpg')]
    cat_files = [f for f in os.listdir(cats_folder) if f.endswith('.jpg')]

    selected_dogs = random.sample(dog_files, min(num_samples, len(dog_files)))
    selected_cats = random.sample(cat_files, min(num_samples, len(cat_files)))



    for filename in selected_dogs:
        shutil.copy(os.path.join(dogs_folder, filename), os.path.join(test_dogs_folder, filename))
    for filename in selected_cats:
        shutil.copy(os.path.join(cats_folder, filename), os.path.join(test_cats_folder, filename))
#ejemplo de uso
test_folder = r"C:\Users\anton\nn\nncatsdogs\data\raw\dogs-vs-cats\test"
create_test_folder(train_folder, test_folder)

In [ ]:
#visualiza 9 fotos de cada clase de la carpeta de test
import matplotlib.pyplot as plt 
import matplotlib.image as mpimg
import random
import os

test_folder = r"C:\Users\anton\nn\nncatsdogs\data\raw\dogs-vs-cats\test"

def visualize_test_samples(test_folder, num_samples=9):
    dog_files = [f for f in os.listdir(test_folder) if f.startswith('dog') and f.endswith('.jpg')]
    cat_files = [f for f in os.listdir(test_folder) if f.startswith('cat') and f.endswith('.jpg')]

    selected_dogs = random.sample(dog_files, num_samples)
    selected_cats = random.sample(cat_files, num_samples)

    plt.figure(figsize=(10, 5))
    
    for i, filename in enumerate(selected_dogs):
        img = plt.imread(os.path.join(test_folder, filename))
        plt.subplot(2, num_samples, i + 1)
        plt.imshow(img)
        plt.title('Dog')
        plt.axis('off')

    for i, filename in enumerate(selected_cats):
        img = plt.imread(os.path.join(test_folder, filename))
        plt.subplot(2, num_samples, num_samples + i + 1)
        plt.imshow(img)
        plt.title('Cat')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

visualize_test_samples(test_folder)

In [ ]:
#cargar los datos usando imagenDataGenerator de keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def load_data(train_folder, test_folder, img_size=(200, 200), batch_size=32):
  
  #crea una imagenDataGenerator para el entrenamiento con aumento de datos
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=40,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    #crea una imagenDataGenerator para el test sin aumento de datos
    test_datagen = ImageDataGenerator(rescale=1./255)

    #carga los datos de entrenamiento y test usando flow_from_directory
    train_generator = train_datagen.flow_from_directory(    
        train_folder,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary'
    )
    test_generator = test_datagen.flow_from_directory(
        test_folder,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary'
    )
    return train_generator, test_generator
#ejemplo de uso
train_generator, test_generator = load_data(train_folder, test_folder)

In [ ]:
#reorganizar archivos en test para que tengan subfolders
import os
import shutil

test_folder = r"C:\Users\anton\nn\nncatsdogs\data\raw\dogs-vs-cats\test"

test_dogs_folder = os.path.join(test_folder, 'dogs')
test_cats_folder = os.path.join(test_folder, 'cats')
os.makedirs(test_dogs_folder, exist_ok=True)
os.makedirs(test_cats_folder, exist_ok=True)

for filename in os.listdir(test_folder):
    if filename.startswith('dog') and filename.endswith('.jpg'):
        shutil.move(os.path.join(test_folder, filename), os.path.join(test_dogs_folder, filename))
    elif filename.startswith('cat') and filename.endswith('.jpg'):
        shutil.move(os.path.join(test_folder, filename), os.path.join(test_cats_folder, filename))

In [ ]:
#copiar archivos de test de vuelta a train para tener datos de entrenamiento
import os
import shutil

train_folder = r"C:\Users\anton\nn\nncatsdogs\data\raw\dogs-vs-cats\train"
test_folder = r"C:\Users\anton\nn\nncatsdogs\data\raw\dogs-vs-cats\test"

train_dogs_folder = os.path.join(train_folder, 'dogs')
train_cats_folder = os.path.join(train_folder, 'cats')
test_dogs_folder = os.path.join(test_folder, 'dogs')
test_cats_folder = os.path.join(test_folder, 'cats')

os.makedirs(train_dogs_folder, exist_ok=True)
os.makedirs(train_cats_folder, exist_ok=True)

for filename in os.listdir(test_dogs_folder):
    if not os.path.exists(os.path.join(train_dogs_folder, filename)):
        shutil.copy(os.path.join(test_dogs_folder, filename), os.path.join(train_dogs_folder, filename))

for filename in os.listdir(test_cats_folder):
    if not os.path.exists(os.path.join(train_cats_folder, filename)):
        shutil.copy(os.path.join(test_cats_folder, filename), os.path.join(train_cats_folder, filename))

In [ ]:
#hacer modelos de cnn con keras usando vgg16 como base
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models 

#carga el modeloo del vgg16 preentrenado sin partes superiores
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(200, 200, 3))    

#crea modelo secuencial y agrega el modelo base
model = models.Sequential()
model.add(base_model)
model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(1, activation='sigmoid'))

#congela las capas del modelo base para no entrenarlas 
base_model.trainable = False
#complila el modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
#resumen del modelo
model.summary()

In [ ]:
#hacer predicciones con el modelo
#entrena el modelo
history = model.fit(train_generator, epochs=1, validation_data=test_generator)

#hacer predicciones con el modelo entrenado
predictions = model.predict(test_generator)
#medir la precision del modelo
from sklearn.metrics import accuracy_score
y_true = test_generator.classes
y_pred = (predictions > 0.5).astype(int).flatten()
accuracy = accuracy_score(y_true, y_pred)
print(f'Accuracy: {accuracy:.4f}')